In [6]:
from bs4 import BeautifulSoup
import requests

URL='https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1'
headers={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0'
}

In [ ]:
# fetch,parse

# soup.select('div.ss_book_box')
def fetch(page):
    params={
        'page':page
    } # 아래 순위 버튼 눌렀을 때 생기는 파라미터 확인
    res=requests.get(URL,params=params,headers=headers,timeout=10)
    print(f'접속 상태:{res.status_code}')
    return res.text

# box.select_one('항목').text.strip()
def picker(tag,selector):
    a=tag.select_one(selector)
    return a.text.strip() if a else ''


def parse(html):
    soup=BeautifulSoup(html,'html.parser')
    boxes=soup.select('div.ss_book_box')
    rows=[]
    for box in boxes:
        # left_cover와 front_cover 두 이미지로 나뉘어있음
        # left_img=box.select_one('img.left_cover')
        # front_img=box.select_one('img.front_cover')
        imgs = box.select("img")
        image_urls = [img.get("src", "") for img in imgs]    # 이미지 여러개 수집
    
        rows.append({
            '카테고리':picker(box,'span.tit_category'),
            '제목':picker(box,'a.bo3'),
            '저자':picker(box,'li:nth-of-type(3)'),
            # Claude 왈 nth-of-type(N) — 
            # 같은 부모 안에서, 지정한 태그(li)와 같은 태그를 가진 형제 요소들만 따로 순서를 매긴 뒤 N번째 요소를 선택합니다. 
            # 이때 순서는 1부터 시작합니다(0이 아님).
            '정가':picker(box,'li:nth-of-type(4)>span'),
            '할인가':picker(box,'span.ss_p2'),
            # '이미지':front_img.get('src','') if front_img else '',
            '이미지':image_urls,
        })
    return rows

In [22]:
result=[]
for i in range(1,11): # i:페이지
    rows=parse(fetch(i))
    result.extend(rows)
    print(f'{1+(50*(i-1))}위~{i*50}위 확인')


접속 상태:200
1위~50위 확인
접속 상태:200
51위~100위 확인
접속 상태:200
101위~150위 확인
접속 상태:200
151위~200위 확인
접속 상태:200
201위~250위 확인
접속 상태:200
251위~300위 확인
접속 상태:200
301위~350위 확인
접속 상태:200
351위~400위 확인
접속 상태:200
401위~450위 확인
접속 상태:200
451위~500위 확인


In [23]:
import pandas as pd

pd.DataFrame(result).to_csv('aladin_bestseller.csv',index=False,encoding='cp949')
print(result[1])

{'카테고리': '[국내도서]', '제목': '그랬다고 적었다', '저자': '김애란 (지은이) | 문학동네 | 2026년 8월', '정가': '17,000', '할인가': '15,300원', '이미지': ['https://image.aladin.co.kr/product/40019/36/SpineShelf/K742130236_d.jpg', 'https://image.aladin.co.kr/product/40019/36/cover200/k742130236_1.jpg']}
